# 第 1 课 · 你的第一个 GET 请求

> **内核提醒（每课都要看）**：本 notebook 必须在 conda 环境 **ai_learn** 中运行。
> 右上角内核应显示 `Python 3.10.21 ('ai_learn')`；打开后从上到下逐格运行（Shift+Enter）。
> 若报 `ModuleNotFoundError`，先检查内核是否选成了 ai_learn。

> 本课只做一件事：用程序向服务器要一次数据，并看懂返回了什么。
> 只引入三个新概念：**请求—响应、状态码、响应文本**。

## 1.0 先解决你最大的困惑：“装了 requests，为什么还要 import？”

这是**两个不同的动作，发生在不同时间**：

1. **`pip install requests`（一次性）**：把这个库的文件下载、安装到环境里。像“把一本书买回来放书架”，装一次永久在。
2. **`import requests`（每个 notebook / 程序开头都要写）**：告诉当前程序“我要用书架上那本书了”。像“每次写作业前把书翻开”。

为什么每次都要 import？因为每个程序 / notebook 都是**独立运行**的，它启动时内存是空的，不知道你要用哪些库。

**用 C++ 类比**：`pip install` 像把第三方库的头文件和源码装到电脑；`import` 像每个 `.cpp` 开头的 `#include` + `using`，缺一不可。

> 你昨天的“没引入 requests”困惑，本质就是：库已经装好了（书架上有书），但这个 notebook 开头还没写 `import`（没把书翻开）。

In [1]:
# 每个 notebook 开头的标准动作：把这一课要用的库“翻开”
import requests                 # 引入 HTTP 请求库（每本 notebook 都要重新 import 一次）
import sys
print('requests 版本：', requests.__version__)
print('当前内核：', sys.executable)      # 必须看到路径里有 \envs\ai_learn\python.exe

requests 版本： 2.34.2
当前内核： C:\Users\Public\miniconda3\envs\ai_learn\python.exe


## 1.1 先看“本地版”：服务器返回的东西，本质就是一段文本

在发真实网络请求前，先打开本地夹具 HTML。它和服务器会返回的内容，是**同一种字符串**。

In [2]:
from pathlib import Path
DATA_DIR = Path(r'C:\Users\千禧\Doubao\chats\2026-09-23\new-chat\AI学习\08_爬虫\data')
html_text = (DATA_DIR / 'books_index.html').read_text(encoding='utf-8')
print('类型：', type(html_text))       # <class 'str'>
print('字符数：', len(html_text))
print('前 200 个字符：')
print(html_text[:200])
# 记住这个感觉：等会儿 requests 从服务器拿到的，也是这样一个 str。

类型： <class 'str'>
字符数： 2398
前 200 个字符：
<!DOCTYPE html>
<html lang="zh-CN">
<head>
    <meta charset="UTF-8">
    <title>在线书店 · 全部书籍（第 1 页）</title>
    <style>
        body { font-family: "Microsoft YaHei", sans-serif; margin: 2rem; backgro


## 1.2 发第一个真实请求：`requests.get(网址)`

- **是什么**：`requests.get(url)` 替你完成“向服务器发请求、拿回响应”这一整套动作。
- 它返回一个 **Response（响应）对象**，服务器给你的所有东西都装在里面。
- 网络偶尔可能不通，所以用 `try/except` 包起来：即使断网，notebook 也不会报错中断。

In [3]:
url = 'https://books.toscrape.com/'     # Scrapy 官方提供的练习网站，稳定且允许抓取
response = None
try:
    response = requests.get(url, timeout=10)   # 发出 GET 请求，最多等 10 秒
    print('请求成功，响应对象类型：', type(response))
except Exception as e:
    print('这次网络没连上（不影响学习）：', type(e).__name__, e)

请求成功，响应对象类型： <class 'requests.models.Response'>


## 1.3 状态码：服务器先告诉你“事情办得怎么样”

`response.status_code` 是一个三位数：

| 状态码 | 含义 |
|---|---|
| **200** | 成功（最想看到） |
| 403 | 拒绝你访问（常见于没伪装浏览器） |
| 404 | 页面不存在 |
| 500 | 服务器自己出问题了 |

类比客服回执：`2` 开头=好，`4` 开头=你的请求有问题，`5` 开头=服务器的问题。
`response.ok` 在状态码 200~299 时为 `True`。

In [4]:
if response is not None:
    print('状态码：', response.status_code)
    print('response.ok：', response.ok)

状态码： 200
response.ok： True


## 1.4 响应体：服务器返回的 HTML 文本

`response.text` 就是网页 HTML（一个 `str`），和 1.1 本地读到的是同一种东西。**这就是爬虫拿到的原材料。**

In [5]:
if response is not None:
    print('响应文本长度：', len(response.text))
    print('前 200 个字符：')
    print(response.text[:200])
# 现在不用看懂它。第 6 课起，我们专门学怎么从这堆文本里把数据抠出来。

响应文本长度： 51294
前 200 个字符：
<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if I


## 1.5 小结（记住这三句）

1. `requests.get(url)` 发请求，返回一个 **Response**。
2. `response.status_code` / `response.ok` 看是否成功（**200 最好**）。
3. `response.text` 是服务器返回的 **HTML 字符串**，是要处理的原材料。

## 1.6 随堂小练习（自己动手，空格里写）

1. 请求一个**不存在的路径** `https://books.toscrape.com/abcdef.html`，打印它的状态码，看看是多少。
2. 请求 `https://quotes.toscrape.com/`，打印 `response.text` 的长度。

In [6]:
# TODO：自己写代码完成上面两题（这是随堂小练，没有单独答案文件）


## 来源与延伸

- requests 官方快速上手：https://requests.readthedocs.io/en/latest/user/quickstart/
- HTTP 状态码大全（MDN）：https://developer.mozilla.org/zh-CN/docs/Web/HTTP/Status
- 官方练习站 books.toscrape：https://books.toscrape.com/
- 下一课：把网址拆开看，学会用“查询参数”向服务器传条件。